In [1]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from datetime import datetime
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment

In [3]:
# Sample KYC Customer Data
data = {
    'CustomerID': ['C001','C002','C003','C004','C005',
                   'C006','C007','C008','C009','C010'],
    'CustomerName': ['John Murphy','Ali Hassan','Global Tech Ltd',
                     'Maria Santos','Black Sea Trading','Emma Clarke',
                     'North Star LLC','David Chen','Patrick Walsh',
                     'Gulf Investments'],
    'Country': ['Ireland','Iran','Ireland','Brazil','Russia',
                'Ireland','Cayman Islands','China','Ireland','UAE'],
    'EntityType': ['Individual','Individual','Corporate','Individual',
                   'Corporate','Individual','Corporate','Individual',
                   'Individual','Corporate'],
    'TransactionAmount': [4500,48000,12000,9500,75000,
                          3000,95000,15000,2000,55000],
    'PEP_Flag': ['No','Yes','No','No','Yes',
                 'No','No','No','No','Yes'],
    'SanctionsFlag': ['No','Yes','No','No','Yes',
                      'No','Yes','No','No','No'],
    'NegativeNews': ['No','Yes','No','Yes','Yes',
                     'No','Yes','No','No','Yes']
}

# Create DataFrame
df = pd.DataFrame(data)
print("Customer data created successfully!")
print(df)

Customer data created successfully!
  CustomerID       CustomerName         Country  EntityType  \
0       C001        John Murphy         Ireland  Individual   
1       C002         Ali Hassan            Iran  Individual   
2       C003    Global Tech Ltd         Ireland   Corporate   
3       C004       Maria Santos          Brazil  Individual   
4       C005  Black Sea Trading          Russia   Corporate   
5       C006        Emma Clarke         Ireland  Individual   
6       C007     North Star LLC  Cayman Islands   Corporate   
7       C008         David Chen           China  Individual   
8       C009      Patrick Walsh         Ireland  Individual   
9       C010   Gulf Investments             UAE   Corporate   

   TransactionAmount PEP_Flag SanctionsFlag NegativeNews  
0               4500       No            No           No  
1              48000      Yes           Yes          Yes  
2              12000       No            No           No  
3               9500       No     

In [4]:
# Define High Risk Countries (based on FATF watchlist)
high_risk_countries = [
    'Iran', 'Russia', 'Cayman Islands', 
    'North Korea', 'Myanmar', 'Syria'
]

print("High risk countries defined:")
print(high_risk_countries)

High risk countries defined:
['Iran', 'Russia', 'Cayman Islands', 'North Korea', 'Myanmar', 'Syria']


In [5]:
# Risk Scoring Function
def calculate_risk_score(row):
    score = 0
    reasons = []
    
    # Country risk — 30 points
    if row['Country'] in high_risk_countries:
        score += 30
        reasons.append('High Risk Country')
    
    # Transaction amount risk — 25 points
    if row['TransactionAmount'] > 10000:
        score += 25
        reasons.append('Large Transaction')
    
    # PEP flag — 20 points
    if row['PEP_Flag'] == 'Yes':
        score += 20
        reasons.append('PEP Identified')
    
    # Sanctions flag — 20 points
    if row['SanctionsFlag'] == 'Yes':
        score += 20
        reasons.append('Sanctions Match')
    
    # Negative news — 5 points
    if row['NegativeNews'] == 'Yes':
        score += 5
        reasons.append('Negative News')
    
    return score, ', '.join(reasons)

# Apply scoring to all customers
df[['RiskScore', 'RiskReasons']] = df.apply(
    lambda row: pd.Series(calculate_risk_score(row)), axis=1
)

# Assign Risk Rating
def assign_rating(score):
    if score >= 50:
        return 'HIGH RISK'
    elif score >= 25:
        return 'MEDIUM RISK'
    else:
        return 'LOW RISK'

df['RiskRating'] = df['RiskScore'].apply(assign_rating)

print("Risk scores calculated!")
print(df[['CustomerName','RiskScore','RiskRating','RiskReasons']])

Risk scores calculated!
        CustomerName  RiskScore   RiskRating  \
0        John Murphy          0     LOW RISK   
1         Ali Hassan        100    HIGH RISK   
2    Global Tech Ltd         25  MEDIUM RISK   
3       Maria Santos          5     LOW RISK   
4  Black Sea Trading        100    HIGH RISK   
5        Emma Clarke          0     LOW RISK   
6     North Star LLC         80    HIGH RISK   
7         David Chen         25  MEDIUM RISK   
8      Patrick Walsh          0     LOW RISK   
9   Gulf Investments         50    HIGH RISK   

                                         RiskReasons  
0                                                     
1  High Risk Country, Large Transaction, PEP Iden...  
2                                  Large Transaction  
3                                      Negative News  
4  High Risk Country, Large Transaction, PEP Iden...  
5                                                     
6  High Risk Country, Large Transaction, Sanction...  
7      

In [7]:
# Generate formatted Excel compliance report
def generate_compliance_report(df):
    
    # Create filename with today's date
    today = datetime.today().strftime('%Y-%m-%d')
    filename = f'KYC_Compliance_Report_{today}.xlsx'
    
    # Create Excel writer
    writer = pd.ExcelWriter(filename, engine='openpyxl')
    df.to_excel(writer, index=False, sheet_name='KYC Report')
    
    # Get workbook and sheet
    workbook = writer.book
    worksheet = writer.sheets['KYC Report']
    
    # Find RiskRating column index automatically
    header_row = [cell.value for cell in worksheet[1]]
    risk_col_index = header_row.index('RiskRating')
    print(f"RiskRating found at column index: {risk_col_index}")
    
    # Define colours
    red_fill = PatternFill(start_color='FF0000', 
                           end_color='FF0000', fill_type='solid')
    amber_fill = PatternFill(start_color='FFA500', 
                             end_color='FFA500', fill_type='solid')
    green_fill = PatternFill(start_color='90EE90', 
                             end_color='90EE90', fill_type='solid')
    header_fill = PatternFill(start_color='003366', 
                              end_color='003366', fill_type='solid')
    white_font = Font(color='FFFFFF', bold=True)
    
    # Format header row
    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = white_font
        cell.alignment = Alignment(horizontal='center')
    
    # Colour code rows by risk rating
    for row in worksheet.iter_rows(min_row=2, 
                                    max_row=worksheet.max_row):
        risk_rating = row[risk_col_index].value
        for cell in row:
            if risk_rating == 'HIGH RISK':
                cell.fill = red_fill
            elif risk_rating == 'MEDIUM RISK':
                cell.fill = amber_fill
            elif risk_rating == 'LOW RISK':
                cell.fill = green_fill
    
    # Auto adjust column widths
    for column in worksheet.columns:
        max_length = 0
        for cell in column:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        worksheet.column_dimensions[
            column[0].column_letter].width = max_length + 2
    
    writer.close()
    print(f"Report generated: {filename}")
    return filename

# Run the report generator
filename = generate_compliance_report(df)

RiskRating found at column index: 10
Report generated: KYC_Compliance_Report_2026-05-30.xlsx


In [9]:
# Print compliance summary
print("=" * 50)
print("KYC COMPLIANCE REPORT SUMMARY")
print("=" * 50)
print(f"Report Date: {datetime.today().strftime('%Y-%m-%d')}")
print(f"Total Customers Reviewed: {len(df)}")

high_count = len(df[df['RiskRating']=='HIGH RISK'])
medium_count = len(df[df['RiskRating']=='MEDIUM RISK'])
low_count = len(df[df['RiskRating']=='LOW RISK'])

print(f"HIGH RISK:   {high_count} customers")
print(f"MEDIUM RISK: {medium_count} customers")
print(f"LOW RISK:    {low_count} customers")
print("=" * 50)
print(f"Full report saved to: {filename}")
print("\nHIGH RISK CUSTOMERS REQUIRING IMMEDIATE REVIEW:")

high_risk = df[df['RiskRating']=='HIGH RISK'][
    ['CustomerName','Country','RiskScore','RiskReasons']]
print(high_risk.to_string(index=False))

KYC COMPLIANCE REPORT SUMMARY
Report Date: 2026-05-30
Total Customers Reviewed: 10
HIGH RISK:   4 customers
MEDIUM RISK: 2 customers
LOW RISK:    4 customers
Full report saved to: KYC_Compliance_Report_2026-05-30.xlsx

HIGH RISK CUSTOMERS REQUIRING IMMEDIATE REVIEW:
     CustomerName        Country  RiskScore                                                                          RiskReasons
       Ali Hassan           Iran        100 High Risk Country, Large Transaction, PEP Identified, Sanctions Match, Negative News
Black Sea Trading         Russia        100 High Risk Country, Large Transaction, PEP Identified, Sanctions Match, Negative News
   North Star LLC Cayman Islands         80                 High Risk Country, Large Transaction, Sanctions Match, Negative News
 Gulf Investments            UAE         50                                     Large Transaction, PEP Identified, Negative News


In [1]:
import pandas as pd

# KYC Dashboard Data
data = {
    'CustomerID': ['C001','C002','C003','C004','C005',
                   'C006','C007','C008','C009','C010'],
    'CustomerName': ['John Murphy','Ali Hassan','Global Tech Ltd',
                     'Maria Santos','Black Sea Trading','Emma Clarke',
                     'North Star LLC','David Chen','Patrick Walsh',
                     'Gulf Investments'],
    'Country': ['Ireland','Iran','Ireland','Brazil','Russia',
                'Ireland','Cayman Islands','China','Ireland','UAE'],
    'EntityType': ['Individual','Individual','Corporate','Individual',
                   'Corporate','Individual','Corporate','Individual',
                   'Individual','Corporate'],
    'TransactionAmount': [4500,48000,12000,9500,75000,
                          3000,95000,15000,2000,55000],
    'PEP_Flag': ['No','Yes','No','No','Yes',
                 'No','No','No','No','Yes'],
    'SanctionsFlag': ['No','Yes','No','No','Yes',
                      'No','Yes','No','No','No'],
    'NegativeNews': ['No','Yes','No','Yes','Yes',
                     'No','Yes','No','No','Yes'],
    'RiskScore': [0,100,25,5,100,0,80,25,0,50],
    'RiskRating': ['LOW RISK','HIGH RISK','MEDIUM RISK','LOW RISK',
                   'HIGH RISK','LOW RISK','HIGH RISK','MEDIUM RISK',
                   'LOW RISK','HIGH RISK'],
    'OnboardDate': ['2023-01-15','2023-02-20','2023-03-10',
                    '2023-04-05','2023-05-18','2023-06-22',
                    '2023-07-30','2023-08-14','2023-09-01',
                    '2023-10-12']
}

df = pd.DataFrame(data)

# Save to CSV in your KYC Portfolio Projects folder
df.to_csv(r'C:\Users\Vadivel Moorthy\KYC Portfolio Projects\KYC_Dashboard_Data.csv', 
          index=False)

print("CSV file created successfully!")
print(df)

CSV file created successfully!
  CustomerID       CustomerName         Country  EntityType  \
0       C001        John Murphy         Ireland  Individual   
1       C002         Ali Hassan            Iran  Individual   
2       C003    Global Tech Ltd         Ireland   Corporate   
3       C004       Maria Santos          Brazil  Individual   
4       C005  Black Sea Trading          Russia   Corporate   
5       C006        Emma Clarke         Ireland  Individual   
6       C007     North Star LLC  Cayman Islands   Corporate   
7       C008         David Chen           China  Individual   
8       C009      Patrick Walsh         Ireland  Individual   
9       C010   Gulf Investments             UAE   Corporate   

   TransactionAmount PEP_Flag SanctionsFlag NegativeNews  RiskScore  \
0               4500       No            No           No          0   
1              48000      Yes           Yes          Yes        100   
2              12000       No            No           No      